In [0]:
%sql
-- step1: Create Unity Catalog Objects
CREATE CATALOG IF NOT EXISTS demo_catalog
COMMENT 'Training catalog for Unity Catalog governance';

--step2: Use the catalog
USE CATALOG demo_catalog;

--step3: Create schemas for medallion architecture
CREATE SCHEMA IF NOT EXISTS demo_catalog.bronze
COMMENT 'Bronze layer - raw ingested data';

CREATE SCHEMA IF NOT EXISTS demo_catalog.silver
COMMENT 'Silver layer -- cleaned and validated data';

CREATE SCHEMA IF NOT EXISTS demo_catalog.gold
COMMENT 'Gold layer - business-level aggregates';

-- step4: Show catalogs and schemas
SHOW CATALOGS;
SHOW SCHEMAS IN demo_catalog;



In [0]:
%sql
--Create and Mange Tables

--Create a managed table in Bronze Layer
CREATE TABLE IF NOT EXISTS demo_catalog.bronze.sales_raw(
transaction_id STRING,
customer_id STRING,
product_id STRING,
amount DECIMAL(10,2),
transaction_date DATE,
region STRING,
ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Raw sales transactions';

-- Insert sample data
INSERT INTO demo_catalog.bronze.sales_raw
VALUES
  ('T001', 'C1001', 'P2001', 129.99, '2024-01-15', 'North', '2024-01-15 08:30:00'),
  ('T002', 'C1002', 'P2002',  49.50, '2024-01-15', 'South', '2024-01-15 09:15:00'),
  ('T003', 'C1003', 'P2003', 899.00, '2024-01-16', 'East',  '2024-01-16 10:45:00'),
  ('T004', 'C1001', 'P2004',  12.75, '2024-01-16', 'North', '2024-01-16 14:20:00'),
  ('T005', 'C1004', 'P2001', 129.99, '2024-01-17', 'West',  '2024-01-17 11:05:00'),
  ('T006', 'C1005', 'P2005', 350.25, '2024-01-17', 'South', '2024-01-17 13:30:00'),
  ('T007', 'C1002', 'P2003', 899.00, '2024-01-18', 'East',  '2024-01-18 09:50:00'),
  ('T008', 'C1006', 'P2002',  49.50, '2024-01-18', 'West',  '2024-01-18 16:10:00');

-- Query the table
SELECT * FROM demo_catalog.bronze.sales_raw

In [0]:
%sql
-- Display detailed metadata and schema information for the bronze sales_raw table,
-- including column names, data types, partition columns, table properties, and location
DESCRIBE EXTENDED demo_catalog.bronze.sales_raw;

In [0]:
# Pyspark: Read and Verify Table

#Read table using pyspark
df = spark.table("demo_catalog.bronze.sales_raw")

#Display Data
print("\n Sample Data:")
display(df)

#Show schema
print("\n Table Schema:")
df.printSchema()

#Basic statistics
print("\n Basic Statistics:")
display(df.describe())

#Count
print("\n Count:")
print(df.count())

#Distinct
print("\n Distinct:")
print(df.distinct().count())

#Count by column
print("\n Count by column:")
print(f"Total Records: {df.count()}")
print(f"Total Revenue: ${df.selectExpr('sum(amount) as total_revenue').collect()[0]['total_revenue']}")

In [0]:
%sql
-- Create silver layer table: cleaned and validated sales data
CREATE OR REPLACE TABLE demo_catalog.silver.sales_cleaned
(
  transaction_id   STRING,
  customer_id      STRING,
  product_id       STRING,
  amount           DECIMAL(10,2),
  transaction_date DATE,
  region           STRING,
  ingestion_timestamp TIMESTAMP,
  is_valid         BOOLEAN,
  cleaned_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Silver layer - cleaned and validated sales transactions';

-- Populate silver table from bronze with validation rules
INSERT INTO demo_catalog.silver.sales_cleaned
SELECT
  transaction_id,
  customer_id,
  product_id,
  amount,
  transaction_date,
  TRIM(region) AS region,
  ingestion_timestamp,
  -- Validate: amount must be positive and all key fields non-null
  (amount > 0
    AND transaction_id IS NOT NULL
    AND customer_id  IS NOT NULL
    AND product_id   IS NOT NULL) AS is_valid,
  current_timestamp() AS cleaned_timestamp
FROM demo_catalog.bronze.sales_raw;

-- Query the silver table
SELECT * FROM demo_catalog.silver.sales_cleaned;

In [0]:
# Basic Access Control in Unity Catalog
# Unity Catalog uses SQL GRANT/REVOKE statements to manage privileges
# on catalogs, schemas, and tables.

# --- Granting Privileges ---
# Common privileges: SELECT, INSERT, UPDATE, DELETE, CREATE, USAGE, ALL PRIVILEGES

# Grant USAGE on the catalog and schema so a principal can browse them
spark.sql("""
  GRANT USE SCHEMA ON SCHEMA demo_catalog.silver TO `account-users`
""")

# Grant SELECT (read-only) on the bronze table
spark.sql("""
  GRANT SELECT ON TABLE demo_catalog.bronze.sales_raw TO `account-users`
""")

# Grant SELECT + INSERT on the silver table (read + write)
spark.sql("""
  GRANT SELECT, INSERT ON TABLE demo_catalog.silver.sales_cleaned TO `data-engineers`
""")

# --- Viewing Granted Privileges ---
# Show all grants for a specific principal
print("Grants for 'account-users':")
display(spark.sql("SHOW GRANTS TO `account-users`"))

# Show grants on a specific object
print("\nGrants on demo_catalog.bronze.sales_raw:")
display(spark.sql("SHOW GRANTS ON TABLE demo_catalog.bronze.sales_raw"))

# --- Revoking Privileges ---
# Revoke INSERT from silver if a principal should become read-only
spark.sql("""
  REVOKE INSERT ON TABLE demo_catalog.silver.sales_cleaned FROM `data-engineers`
""")

# Verify the change
print("\nGrants on silver.sales_cleaned after revoke:")
display(spark.sql("SHOW GRANTS ON TABLE demo_catalog.silver.sales_cleaned"))

# --- Key Points ---
# 1. USE SCHEMA / USE CATALOG privileges are required to browse objects.
# 2. Granularity: you can grant at catalog, schema, or table level.
# 3. GRANT ALL PRIVILEGES gives every applicable privilege on an object.
# 4. Principals can be individual users, groups, or service principals.
# 5. For production, prefer group-based grants over user-based grants.

In [0]:
%sql

-- =============================================================================
-- UNITY CATALOG GOVERNANCE: STEP-BY-STEP IMPLEMENTATION
-- =============================================================================
-- This cell demonstrates a complete governance workflow:
--   1) Create organization structure (catalog + schemas)
--   2) Create data assets (tables with sample data)
--   3) Configure access control (GRANT / REVOKE)
--   4) Validate permissions (SHOW GRANTS + information_schema)
--   5) Document and monitor (tags, comments, system tables)
-- =============================================================================


-- ═══════════════════════════════════════════════════════════════════════════
-- STEP 1: CREATE ORGANIZATION STRUCTURE
-- ═══════════════════════════════════════════════════════════════════════════
-- Build the catalog and medallion-style schemas that mirror business units.

-- 1a. Create a catalog for the governance demo
CREATE CATALOG IF NOT EXISTS gov_demo
COMMENT 'Governance demo catalog — org structure, access control, monitoring';

-- 1b. Switch to the catalog so subsequent unqualified schema refs resolve here
USE CATALOG gov_demo;

-- 1c. Create schemas for each medallion layer
CREATE SCHEMA IF NOT EXISTS gov_demo.bronze
COMMENT 'Bronze layer — raw ingested data, no transformations';

CREATE SCHEMA IF NOT EXISTS gov_demo.silver
COMMENT 'Silver layer — cleaned, validated, conformed data';

CREATE SCHEMA IF NOT EXISTS gov_demo.gold
COMMENT 'Gold layer — business-level aggregates and curated datasets';

-- 1d. Verify the structure
SHOW SCHEMAS IN gov_demo;


-- ═══════════════════════════════════════════════════════════════════════════
-- STEP 2: CREATE DATA ASSETS
-- ═══════════════════════════════════════════════════════════════════════════
-- Create tables in each layer and populate them with sample data.

-- 2a. Bronze: raw sales transactions
CREATE TABLE IF NOT EXISTS gov_demo.bronze.sales_raw (
  transaction_id      STRING,
  customer_id         STRING,
  product_id          STRING,
  amount              DECIMAL(10,2),
  transaction_date    DATE,
  region              STRING,
  ingestion_timestamp TIMESTAMP
) USING DELTA
COMMENT 'Raw sales transactions ingested from source systems';

INSERT INTO gov_demo.bronze.sales_raw
VALUES
  ('T001','C1001','P2001', 129.99,'2024-01-15','North','2024-01-15 08:30:00'),
  ('T002','C1002','P2002',  49.50,'2024-01-15','South','2024-01-15 09:15:00'),
  ('T003','C1003','P2003', 899.00,'2024-01-16','East', '2024-01-16 10:45:00'),
  ('T004','C1001','P2004',  12.75,'2024-01-16','North','2024-01-16 14:20:00'),
  ('T005','C1004','P2001', 129.99,'2024-01-17','West', '2024-01-17 11:05:00'),
  ('T006','C1005','P2005', 350.25,'2024-01-17','South','2024-01-17 13:30:00');

-- 2b. Silver: cleaned and validated sales data
CREATE OR REPLACE TABLE gov_demo.silver.sales_cleaned
USING DELTA
COMMENT 'Silver layer — validated sales transactions with data quality flags'
AS
SELECT
  transaction_id,
  customer_id,
  product_id,
  amount,
  transaction_date,
  TRIM(region) AS region,
  ingestion_timestamp,
  (amount > 0
    AND transaction_id IS NOT NULL
    AND customer_id  IS NOT NULL
    AND product_id   IS NOT NULL) AS is_valid,
  current_timestamp() AS cleaned_timestamp
FROM gov_demo.bronze.sales_raw;

-- 2c. Gold: revenue summary by region
CREATE OR REPLACE TABLE gov_demo.gold.sales_by_region
USING DELTA
COMMENT 'Gold layer — total revenue and transaction count by region'
AS
SELECT
  region,
  COUNT(*)            AS transaction_count,
  SUM(amount)         AS total_revenue
FROM gov_demo.silver.sales_cleaned
WHERE is_valid = true
GROUP BY region;

-- 2d. Verify data assets
SELECT 'bronze' AS layer, COUNT(*) AS row_count FROM gov_demo.bronze.sales_raw
UNION ALL
SELECT 'silver',       COUNT(*)                  FROM gov_demo.silver.sales_cleaned
UNION ALL
SELECT 'gold',          COUNT(*)                  FROM gov_demo.gold.sales_by_region;


-- ═══════════════════════════════════════════════════════════════════════════
-- STEP 3: CONFIGURE ACCESS CONTROL
-- ═══════════════════════════════════════════════════════════════════════════
-- Grant privileges to groups at the appropriate granularity.
-- Replace group names with real principals in your workspace.

-- 3a. Grant catalog-level USAGE so groups can browse the catalog
GRANT USE CATALOG ON CATALOG gov_demo TO `account-users`;

-- 3b. Grant schema-level USAGE so groups can browse each schema
GRANT USE SCHEMA ON SCHEMA gov_demo.bronze TO `account-users`;
GRANT USE SCHEMA ON SCHEMA gov_demo.silver TO `account-users`;
GRANT USE SCHEMA ON SCHEMA gov_demo.gold   TO `account-users`;

-- 3c. Bronze: read-only for analysts
GRANT SELECT ON TABLE gov_demo.bronze.sales_raw TO `account-users`;

-- 3d. Silver: read + write for data engineers
GRANT SELECT, INSERT ON TABLE gov_demo.silver.sales_cleaned TO `data-engineers`;

-- 3e. Gold: read-only for business analysts, full access for admins
GRANT SELECT ON TABLE gov_demo.gold.sales_by_region TO `account-users`;
GRANT ALL PRIVILEGES ON TABLE gov_demo.gold.sales_by_region TO `admin`;

-- 3f. Revoke INSERT from a group that should become read-only
REVOKE INSERT ON TABLE gov_demo.silver.sales_cleaned FROM `data-engineers`;


-- ═══════════════════════════════════════════════════════════════════════════
-- STEP 4: VALIDATE PERMISSIONS
-- ═══════════════════════════════════════════════════════════════════════════
-- Inspect grants to confirm access is configured correctly.

-- 4a. Show all grants on each object
SHOW GRANTS ON CATALOG gov_demo;
SHOW GRANTS ON TABLE gov_demo.bronze.sales_raw;
SHOW GRANTS ON TABLE gov_demo.silver.sales_cleaned;
SHOW GRANTS ON TABLE gov_demo.gold.sales_by_region;

-- 4b. Show all grants for a specific principal
SHOW GRANTS TO `account-users`;

-- 4c. Query the information_schema for a programmatic view of privileges
SELECT
  table_catalog,
  table_schema,
  table_name,
  privilege_type,
  grantee
FROM system.information_schema.table_privileges
WHERE table_catalog = 'gov_demo'
ORDER BY table_schema, table_name, privilege_type;


-- ═══════════════════════════════════════════════════════════════════════════
-- STEP 5: DOCUMENT AND MONITOR
-- ═══════════════════════════════════════════════════════════════════════════
-- Tag assets for classification, add descriptive comments, and set up
-- monitoring queries against system tables.

-- 5a. Tag the catalog with business metadata
SET TAG ON CATALOG gov_demo business_unit = `retail`;
SET TAG ON CATALOG gov_demo environment   = `demo`;

-- 5b. Tag schemas by medallion layer
SET TAG ON SCHEMA gov_demo.bronze layer = bronze;
SET TAG ON SCHEMA gov_demo.bronze quality_tier = raw;
SET TAG ON SCHEMA gov_demo.silver layer = silver;
SET TAG ON SCHEMA gov_demo.silver quality_tier = validated;
SET TAG ON SCHEMA gov_demo.gold layer = gold;
SET TAG ON SCHEMA gov_demo.gold quality_tier = curated;

-- 5c. Tag tables with ownership and SLA metadata
SET TAG ON TABLE gov_demo.bronze.sales_raw        owner = `data-engineering`;
SET TAG ON TABLE gov_demo.bronze.sales_raw        sla_tier = bronze;
SET TAG ON TABLE gov_demo.silver.sales_cleaned    owner = `data-engineering`;
SET TAG ON TABLE gov_demo.silver.sales_cleaned    sla_tier = silver;
SET TAG ON TABLE gov_demo.gold.sales_by_region    owner = `analytics`;
SET TAG ON TABLE gov_demo.gold.sales_by_region    sla_tier = gold;

-- 5d. Tag sensitive columns (PII classification)
SET TAG ON COLUMN gov_demo.bronze.sales_raw.customer_id pii = true;
SET TAG ON COLUMN gov_demo.bronze.sales_raw.customer_id sensitivity = medium;
SET TAG ON COLUMN gov_demo.silver.sales_cleaned.customer_id pii = true;
SET TAG ON COLUMN gov_demo.silver.sales_cleaned.customer_id sensitivity = medium;

-- 5e. Add descriptive comments on key columns
COMMENT ON COLUMN gov_demo.bronze.sales_raw.transaction_id IS
  'Unique identifier for each sales transaction. Format: T followed by digits.';
COMMENT ON COLUMN gov_demo.bronze.sales_raw.customer_id IS
  'Customer identifier. Tagged as PII — may require masking in shared contexts.';
COMMENT ON COLUMN gov_demo.bronze.sales_raw.amount IS
  'Transaction amount in USD. Must be positive for valid records.';

COMMENT ON COLUMN gov_demo.silver.sales_cleaned.is_valid IS
  'Data quality flag: true when amount > 0 and all key fields are non-null.';

COMMENT ON COLUMN gov_demo.gold.sales_by_region.total_revenue IS
  'Sum of valid transaction amounts per region in USD.';

-- 5f. Query table-level tags across the catalog
SELECT
  catalog_name,
  schema_name,
  table_name,
  tag_name,
  tag_value
FROM system.information_schema.table_tags
WHERE catalog_name = 'gov_demo'
ORDER BY schema_name, table_name, tag_name;

-- 5g. Query column-level PII tags
SELECT
  catalog_name,
  schema_name,
  table_name,
  column_name,
  tag_name,
  tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'gov_demo'
  AND tag_name = 'pii'
ORDER BY schema_name, table_name, column_name;

-- 5h. Monitor: recent queries against this catalog's tables (last 24h)
SELECT
  start_time,
  statement_text,
  statement_type,
  total_elapsed_ms,
  rows_produced,
  executed_by
FROM system.query.history
WHERE statement_text LIKE '%gov_demo%'
  AND start_time >= current_timestamp() - INTERVAL 24 HOURS
ORDER BY start_time DESC
LIMIT 50;

-- 5i. Monitor: table lineage to understand data flow into gold layer
SELECT
  source_table_full_name,
  target_table_full_name,
  created_by,
  created_at
FROM system.access.table_lineage
WHERE target_table_full_name LIKE '%gov_demo.gold%'
ORDER BY created_at DESC
LIMIT 20;

-- 5j. Monitor: audit who accessed or modified the catalog's objects
SELECT
  event_time,
  action_name,
  event_type,
  user_identity.email AS actor,
  request_params
FROM system.access.audit
WHERE service_name = 'unityCatalog'
  AND request_params.catalog_name = 'gov_demo'
  AND event_date >= current_date() - INTERVAL 1 DAY
ORDER BY event_time DESC
LIMIT 50;


-- =============================================================================
-- SUMMARY
-- =============================================================================
-- Step 1  → Created catalog `gov_demo` with bronze/silver/gold schemas.
-- Step 2  → Created tables in each layer and populated with sample data.
-- Step 3  → Granted USAGE + SELECT/INSERT privileges to groups; revoked
--           INSERT on silver to enforce read-only for data-engineers.
-- Step 4  → Validated with SHOW GRANTS and information_schema.table_privileges.
-- Step 5  → Tagged assets (ownership, SLA, PII), added column comments,
--           and set up monitoring via system.query.history, table_lineage,
--           and system.access.audit.
-- =============================================================================
-- ============================================================
-- Gold Layer: Business-Level Aggregates
-- ============================================================

-- Step 1: Create the gold table for sales aggregated by region and date
CREATE OR REPLACE TABLE demo_catalog.gold.sales_by_region_daily
(
  transaction_date DATE,
  region           STRING,
  total_sales      DECIMAL(18,2),
  num_transactions BIGINT,
  avg_sale_amount  DECIMAL(18,2),
  gold_timestamp   TIMESTAMP
)
USING DELTA
COMMENT 'Gold layer - daily sales summary by region';

-- Step 2: Populate the gold table from the silver layer
-- Only include records that passed validation (is_valid = true)
INSERT INTO demo_catalog.gold.sales_by_region_daily
SELECT
  transaction_date,
  TRIM(region) AS region,
  CAST(SUM(amount)        AS DECIMAL(18,2)) AS total_sales,
  COUNT(*)                                       AS num_transactions,
  CAST(AVG(amount)        AS DECIMAL(18,2)) AS avg_sale_amount,
  current_timestamp()                            AS gold_timestamp
FROM demo_catalog.silver.sales_cleaned
WHERE is_valid = true
GROUP BY transaction_date, TRIM(region)
ORDER BY transaction_date, region;

-- Step 3: Query the gold table
SELECT * FROM demo_catalog.gold.sales_by_region_daily
ORDER BY transaction_date, region;


In [0]:
%sql

-- ============================================================
-- MEDALLION ARCHITECTURE: Bronze → Silver → Gold
-- ============================================================
-- The medallion architecture organizes data in three layers:
--   Bronze  : Raw data as-is from source systems (no transformation)
--   Silver  : Cleaned, validated, conformed data (light transformation)
--   Gold    : Business-level aggregates and curated datasets
-- ============================================================

-- ═══════════════════════════════════════════════════════════
-- BRONZE LAYER — Raw ingested data
-- ═══════════════════════════════════════════════════════════
CREATE TABLE IF NOT EXISTS demo_catalog.bronze.orders_raw (
  order_id            STRING,
  customer_id         STRING,
  product_id          STRING,
  quantity            INT,
  unit_price           DECIMAL(10,2),
  order_date          DATE,
  store_region        STRING,
  ingestion_timestamp TIMESTAMP
) USING DELTA
COMMENT 'Bronze layer - raw order data ingested from source systems';

INSERT INTO demo_catalog.bronze.orders_raw
VALUES
  ('O001', 'C1001', 'P2001', 2,  64.99, '2024-02-01', 'North', '2024-02-01 07:15:00'),
  ('O002', 'C1002', 'P2002', 1, 199.00, '2024-02-01', 'South', '2024-02-01 08:45:00'),
  ('O003', 'C1003', 'P2003', 5,  12.50, '2024-02-02', 'East',  '2024-02-02 10:30:00'),
  ('O004', 'C1001', 'P2004', 3,  29.75, '2024-02-02', 'North', '2024-02-02 14:05:00'),
  ('O005', 'C1004', 'P2001', 2,  64.99, '2024-02-03', 'West',  '2024-02-03 09:20:00'),
  ('O006', 'C1005', 'P2005', 1, 449.00, '2024-02-03', 'South', '2024-02-03 11:55:00'),
  ('O007', 'C1002', 'P2003', 4,  12.50, '2024-02-04', 'East',  '2024-02-04 13:40:00'),
  ('O008', 'C1006', 'P2002', 1, 199.00, '2024-02-04', 'West',  '2024-02-04 15:10:00');

SELECT * FROM demo_catalog.bronze.orders_raw;

-- ═══════════════════════════════════════════════════════════
-- SILVER LAYER — Cleaned, validated, conformed data
-- ═══════════════════════════════════════════════════════════
CREATE OR REPLACE TABLE demo_catalog.silver.orders_cleaned
USING DELTA
COMMENT 'Silver layer - validated orders with data quality flags and derived fields'
AS
SELECT
  order_id,
  customer_id,
  product_id,
  quantity,
  unit_price,
  (quantity * unit_price)                       AS total_amount,
  order_date,
  UPPER(TRIM(store_region))                      AS region,
  ingestion_timestamp,
  -- Validation rules
  (quantity > 0
    AND unit_price > 0
    AND order_id    IS NOT NULL
    AND customer_id IS NOT NULL
    AND product_id  IS NOT NULL)                  AS is_valid,
  current_timestamp()                            AS cleaned_timestamp
FROM demo_catalog.bronze.orders_raw;

SELECT * FROM demo_catalog.silver.orders_cleaned;

-- ═══════════════════════════════════════════════════════════
-- GOLD LAYER — Business-level aggregates
-- ═══════════════════════════════════════════════════════════

-- Gold table 1: Daily revenue by region
CREATE OR REPLACE TABLE demo_catalog.gold.orders_daily_by_region
USING DELTA
COMMENT 'Gold layer - daily revenue and order count by region'
AS
SELECT
  order_date,
  region,
  COUNT(*)                                       AS num_orders,
  CAST(SUM(total_amount) AS DECIMAL(18,2))        AS total_revenue,
  CAST(AVG(total_amount) AS DECIMAL(18,2))        AS avg_order_value
FROM demo_catalog.silver.orders_cleaned
WHERE is_valid = true
GROUP BY order_date, region;

SELECT * FROM demo_catalog.gold.orders_daily_by_region
ORDER BY order_date, region;

-- Gold table 2: Customer spending summary
CREATE OR REPLACE TABLE demo_catalog.gold.customer_spending_summary
USING DELTA
COMMENT 'Gold layer - total spending and order count per customer'
AS
SELECT
  customer_id,
  COUNT(*)                                       AS num_orders,
  CAST(SUM(total_amount) AS DECIMAL(18,2))        AS lifetime_value,
  CAST(AVG(total_amount) AS DECIMAL(18,2))        AS avg_order_value,
  MIN(order_date)                               AS first_order_date,
  MAX(order_date)                               AS last_order_date
FROM demo_catalog.silver.orders_cleaned
WHERE is_valid = true
GROUP BY customer_id;

SELECT * FROM demo_catalog.gold.customer_spending_summary
ORDER BY lifetime_value DESC;

-- Gold table 3: Product performance
CREATE OR REPLACE TABLE demo_catalog.gold.product_performance
USING DELTA
COMMENT 'Gold layer - sales performance metrics by product'
AS
SELECT
  product_id,
  COUNT(*)                                       AS num_orders,
  SUM(quantity)                                  AS total_units_sold,
  CAST(SUM(total_amount) AS DECIMAL(18,2))        AS total_revenue,
  CAST(AVG(total_amount) AS DECIMAL(18,2))        AS avg_order_value
FROM demo_catalog.silver.orders_cleaned
WHERE is_valid = true
GROUP BY product_id;

SELECT * FROM demo_catalog.gold.product_performance
ORDER BY total_revenue DESC;

-- ═══════════════════════════════════════════════════════════
-- VERIFY: Row counts across all layers
-- ═══════════════════════════════════════════════════════════
SELECT 'bronze'   AS layer, COUNT(*) AS row_count FROM demo_catalog.bronze.orders_raw
UNION ALL
SELECT 'silver',          COUNT(*)               FROM demo_catalog.silver.orders_cleaned
UNION ALL
SELECT 'gold (daily)',    COUNT(*)               FROM demo_catalog.gold.orders_daily_by_region
UNION ALL
SELECT 'gold (customer)', COUNT(*)               FROM demo_catalog.gold.customer_spending_summary
UNION ALL
SELECT 'gold (product)',  COUNT(*)               FROM demo_catalog.gold.product_performance;
